# Picsart Jobs Scraper

**Source:** [picsart.com/careers](https://picsart.com/careers/jobs)  
**Backend ATS:** Greenhouse (public API, no authentication required)  
**API:** `https://boards-api.greenhouse.io/v1/boards/picsart/jobs?content=true`  
**Filter:** Armenia-based positions only

### Why Picsart separately?
Large Armenian tech companies post openings on their own careers portals rather than on aggregators like LinkedIn, Staff.am, or job.am.

### API structure
- Single GET request returns all live postings in one response (`meta.total` = total count)
- Per-job fields: `id`, `title`, `content` (HTML), `location.name`, `departments[].name`, `absolute_url`, `first_published`
- `content` field contains full HTML job description — stripped to plain text for NLP

### Ethics & robots.txt
- Greenhouse public board API is open by design — no auth required, publicly documented
- `robots.txt` at picsart.com does not restrict `/careers/` paths (checked 2026-03-22)
- Single API call, no HTML page crawling

### Output files
- `data/raw/jobs/picsart_jobs_raw.csv`
- `data/processed/jobs/picsart_jobs_standardized.csv`

In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

API_URL  = "https://boards-api.greenhouse.io/v1/boards/picsart/jobs"
HEADERS  = {"User-Agent": "ThesisResearch/1.0 (Armenian IT curriculum alignment; academic use)"}
PARAMS   = {"content": "true"}

BASE_DIR = Path.cwd()
RAW_DIR  = BASE_DIR / "data" / "raw" / "jobs"
PROC_DIR = BASE_DIR / "data" / "processed" / "jobs"

print(f"API: {API_URL}")
print(f"Raw output:  {RAW_DIR}")
print(f"Proc output: {PROC_DIR}")

API: https://boards-api.greenhouse.io/v1/boards/picsart/jobs
Raw output:  /Users/lianaaghamalyan/thesis_data/data/raw/jobs
Proc output: /Users/lianaaghamalyan/thesis_data/data/processed/jobs


## Helper functions

In [2]:
def html_to_text(html_str):
    """Strip HTML tags and normalize whitespace."""
    if not html_str:
        return ""
    text = BeautifulSoup(str(html_str), "html.parser").get_text("\n", strip=True)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def is_armenia(location_str):
    """Return True if location is in Armenia."""
    loc = (location_str or "").lower()
    return "armenia" in loc or "yerevan" in loc

print("Helpers defined.")

Helpers defined.


## Step 1 — Fetch all jobs from Greenhouse API

In [3]:
resp = requests.get(API_URL, headers=HEADERS, params=PARAMS, timeout=30)
resp.raise_for_status()
data = resp.json()

total_all = data["meta"]["total"]
all_jobs  = data["jobs"]
print(f"Total jobs on board: {total_all}")
print(f"Jobs returned: {len(all_jobs)}")
print()

armenia_jobs = [j for j in all_jobs if is_armenia(j.get("location", {}).get("name", ""))]
print(f"Armenia-based jobs: {len(armenia_jobs)}")
print()
for j in armenia_jobs:
    dept = j["departments"][0]["name"] if j.get("departments") else "—"
    print(f"  [{j['id']}] {j['title']} | {j['location']['name']} | {dept}")

Total jobs on board: 5
Jobs returned: 5

Armenia-based jobs: 2

  [5734906004] Social Media Content Creator (Contractor) | Yerevan, Armenia | Marketing
  [5702841004] User Acquisition Specialist (Service  Contract) | Yerevan, Armenia | Marketing


## Step 2 — Extract fields and build records

In [4]:
records = []
for j in armenia_jobs:
    dept     = ", ".join(d["name"] for d in j.get("departments", []) if d.get("name"))
    offices  = ", ".join(o["name"] for o in j.get("offices", []) if o.get("name"))
    full_text = html_to_text(j.get("content", ""))
    records.append({
        "source":       "picsart",
        "source_url":   j.get("absolute_url", ""),
        "job_id":       j.get("id", ""),
        "job_title":    j.get("title", ""),
        "company_name": "Picsart",
        "location":     j.get("location", {}).get("name", ""),
        "department":   dept,
        "office":       offices,
        "posting_date": (j.get("first_published") or "")[:10],
        "full_text":    full_text,
    })
    print(f"  {j['title']}")
    print(f"    dept: {dept} | posted: {(j.get('first_published') or '')[:10]}")
    print(f"    full_text: {len(full_text)} chars")
    print()

  Social Media Content Creator (Contractor)
    dept: Marketing | posted: 2026-01-23
    full_text: 7865 chars

  User Acquisition Specialist (Service  Contract)
    dept: Marketing | posted: 2025-10-21
    full_text: 7423 chars



## Step 3 — Save outputs

In [5]:
df = pd.DataFrame(records)
raw_path = RAW_DIR / "picsart_jobs_raw.csv"
df.to_csv(raw_path, index=False, encoding="utf-8")
print(f"Raw saved: {raw_path} ({len(df)} rows)")

std_cols = ["source","source_url","job_title","company_name","location","department","posting_date","full_text"]
std_df = df[std_cols]
std_path = PROC_DIR / "picsart_jobs_standardized.csv"
std_df.to_csv(std_path, index=False, encoding="utf-8")
print(f"Standardized saved: {std_path} ({len(std_df)} rows)")
print()
print("=== FIELD COVERAGE ===")
for col in std_cols:
    filled = std_df[col].replace("", pd.NA).notna().sum()
    print(f"  {col:20s}: {filled}/{len(std_df)} ({filled/len(std_df)*100:.0f}%)")
print()
ft = std_df["full_text"].str.len()
print(f"full_text — Min: {ft.min()}  Median: {ft.median():.0f}  Max: {ft.max()}")

Raw saved: /Users/lianaaghamalyan/thesis_data/data/raw/jobs/picsart_jobs_raw.csv (2 rows)
Standardized saved: /Users/lianaaghamalyan/thesis_data/data/processed/jobs/picsart_jobs_standardized.csv (2 rows)

=== FIELD COVERAGE ===
  source              : 2/2 (100%)
  source_url          : 2/2 (100%)
  job_title           : 2/2 (100%)
  company_name        : 2/2 (100%)
  location            : 2/2 (100%)
  department          : 2/2 (100%)
  posting_date        : 2/2 (100%)
  full_text           : 2/2 (100%)

full_text — Min: 7423  Median: 7644  Max: 7865
